# DDL: `dbspend360_total_pipeline_spends`

Creates the final per-pipeline / per-day / per-product spend rollup for declarative "Pipeline Compute", scoped to billing rows
where `usage_metadata.dlt_pipeline_id IS NOT NULL` (any `billing_origin_product` — DLT, DBSQL materialized views, online
tables, vector search, model serving, AI functions).

Sibling of `dbspend360_total_pool_spends`, keyed on `(workspace_id, pipeline_id, usage_date, billing_origin_product)`.
`workspace_id` is in the key because `pipeline_id` is only unique within a single workspace. `billing_origin_product` stays
in the grain so the per-workload `$` split is exact (reconciles row-for-row with the staging table, no within-day
dominant-product approximation). Pipeline metadata (`pipeline_name`, `pipeline_type`, `created_by`, `run_as`) is
denormalized straight from `system.lakeflow.pipelines` (no REST API) so the UI needs no live join.

Derived dimensions:
- `workload_type`  - friendly label mapped from `billing_origin_product` (DLT Pipeline / DBSQL Materialized View / Online
  Table / Vector Search / Model Serving / AI Functions / raw value for unknowns).
- `compute_mode`   - `serverless` / `classic` / `mixed` (derived from `cluster_id` being NULL in staging).
- `cost_basis`     - `full` (serverless, complete cost) / `dbu_only` (classic, excludes cloud VM) / `partial` (mixed).

Three-state, product-aware snapshot handling (per the plan §3.5):
- Active: `metadata_missing = FALSE`, `pipeline_deleted_at IS NULL`.
- Deleted but visible: `metadata_missing = FALSE`, `pipeline_deleted_at` populated (UI renders "Deleted YYYY-MM-DD").
- Metadata not available: `metadata_missing = TRUE`, `pipeline_deleted_at IS NULL` (no `system.lakeflow.pipelines` row —
  the *expected* state for Vector Search / cross-region; UI renders a neutral grey badge; `pipeline_name` falls back to
  `"Pipeline {pipeline_id}"`).

`cloud_cost DOUBLE` carries EC2/EBS for classic pipeline clusters (v2, plan §3.2/§3.3). Classic clusters are tagged
`ClusterId` on AWS, so their cloud cost is already in `dbspend360_cloud_cost_explorer`; each cluster-day cloud is attributed
to its pipeline (DBU-weighted, reconciling instead of double-counting a shared cluster) and then spread across that
pipeline's `billing_origin_product` rows by classic-DBU share. The per-product split is an informed apportionment; the
pipeline-day SUM stays exact and is reconciled to the explorer per `(cluster_id, usage_date, currency)` within `$0.01`.
Serverless rows have **no** separate VM line, so `cloud_cost` stays `NULL` (the UI renders `-` via `compute_mode`, never a
misleading `$0`); classic rows whose explorer cost has not landed yet are also `NULL`. `total_cost = databricks_cost +
COALESCE(cloud_cost, 0)` keeps totals safe in both cases.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_total_pipeline_spends (
  workspace_id            STRING,
  pipeline_id             STRING,
  usage_date              DATE,
  pipeline_name           STRING,
  pipeline_type           STRING,
  created_by              STRING,
  run_as                  STRING,
  workload_type           STRING,
  compute_mode            STRING,
  cost_basis              STRING,
  metadata_missing        BOOLEAN,
  pipeline_deleted_at     TIMESTAMP,
  databricks_cost         DOUBLE,
  update_cost             DOUBLE,
  maintenance_cost        DOUBLE,
  cloud_cost              DOUBLE,
  total_cost              DOUBLE,
  currency                STRING,
  sku_name                STRING,
  billing_origin_product  STRING,
  created_at              TIMESTAMP,
  updated_at              TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_total_pipeline_spends")